# Workshop 2. Confidence intervals

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture02/workshop02.ipynb)

In this workshop you write the code yourselves. Work in your group or alone, and ask the TAs when you are stuck.

Task 1 is the lecture walkthrough in code, fully worked: run it, answer the questions and change one thing. Tasks 2
to 4 apply the same steps to new questions on the RetailCorp data, starting from an empty cell. Then do the short
problems, which use small numbers and no data, much like a question at the oral exam. The extension tasks are for
those who have time left.

Under each task there are three hints: the first names the statistical step, the second names the functions, and the
third gives the structure of the code as comments. Try the task before you open them. The code patterns below, the
slides and the lecture notebook are your main references. You are welcome to use an LLM, but every task can be
solved without one, and you should be able to explain the statistics in your answers yourself.

## Toolbox

The functions you need today. The lecture notebook shows all of them in use.

| Function | What it does |
|---|---|
| `df["col"].mean()`, `.std(ddof=1)`, `.var(ddof=1)` | Mean, standard deviation and variance of a column, dividing by n − 1 |
| `len(x)` | Number of observations |
| `df.loc[df["col"] == "value", "other"]` | The column `other` for the rows where `col` has a given value |
| `(df["col"] > value).mean()` | Share of rows where the column is above a value |
| `np.sqrt(x)` | Square root |
| `stats.norm.ppf(q)` | Value with probability q below it in the standard normal: z<sub>α/2</sub> is `stats.norm.ppf(1 - alpha / 2)` |
| `stats.t.ppf(q, df=n - 1)` | The same for the t distribution with n − 1 degrees of freedom |
| `stats.chi2.ppf(q, df=n - 1)` | The same for the chi-square distribution with n − 1 degrees of freedom |
| `stats.skew(x)` | Skewness |
| `np.ceil(x)` | Round up to the next whole number |
| `plt.hist(x, bins=20)` | Histogram |
| `rng = np.random.default_rng(seed)`, `rng.choice(x, size=n, replace=False)` | Random sample of n values (extension tasks) |

## Code patterns

Three complete patterns for today, one for each interval. Copy a pattern into a task cell, replace `df` and
`"column"` with the data and column you need, and set the confidence level. The patterns are templates; they do not
run as they stand.

**Interval for a mean, σ unknown (t)**

```python
x = df["column"]                            # the variable
n = len(x)
confidence = 0.95
alpha = 1 - confidence
t = stats.t.ppf(1 - alpha / 2, df=n - 1)    # critical value with alpha/2 above it
se = x.std(ddof=1) / np.sqrt(n)             # standard error s / sqrt(n)
me = t * se                                 # margin of error
print(f"Interval: [{x.mean() - me:.2f}, {x.mean() + me:.2f}]")
```

**Interval for a proportion**

```python
x = df["column"]                            # a 0/1 column, or a condition such as df["column"] > value
n = len(x)
confidence = 0.95
alpha = 1 - confidence
p_hat = x.mean()                            # the share of 1s (or of True)
print(n * p_hat * (1 - p_hat))              # the condition: should be above 5
z = stats.norm.ppf(1 - alpha / 2)
me = z * np.sqrt(p_hat * (1 - p_hat) / n)
print(f"Interval: [{p_hat - me:.3f}, {p_hat + me:.3f}]")
```

**Interval for a variance and a standard deviation (needs a normal population)**

```python
x = df["column"]
n = len(x)
confidence = 0.95
alpha = 1 - confidence
s2 = x.var(ddof=1)
chi2_hi = stats.chi2.ppf(1 - alpha / 2, df=n - 1)   # the larger value gives the lower limit
chi2_lo = stats.chi2.ppf(alpha / 2, df=n - 1)       # the smaller value gives the upper limit
lower = (n - 1) * s2 / chi2_hi
upper = (n - 1) * s2 / chi2_lo
print(f"Variance: [{lower:.2f}, {upper:.2f}]")
print(f"Standard deviation: [{np.sqrt(lower):.2f}, {np.sqrt(upper):.2f}]")
```

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
sample = pd.read_csv(url + "orders_sample.csv")          # the study of 200 orders
population = pd.read_csv(url + "orders_population.csv")  # all 50,000 orders, used only in the extension tasks

## Core tasks

### Task 1. The lecture walkthrough in code (worked)

How long does a delivery take on average, across all of RetailCorp's orders? The cell below computes the 95%
confidence interval for the mean delivery time from the 200 orders, with σ unknown, exactly as in the lecture. Run it
and answer the questions.

1. Which number in the output is the margin of error, and from which two numbers is it computed?
2. Change `confidence = 0.95` to `0.99` and run the cell again. Which numbers change and which stay the same?
3. Write the 95% result in one or two sentences for a manager at RetailCorp.

In [ ]:
x = sample["delivery_hours"]
n = len(x)
confidence = 0.95                           # change to 0.99 in question 2
alpha = 1 - confidence
t = stats.t.ppf(1 - alpha / 2, df=n - 1)    # critical value with alpha/2 above it
se = x.std(ddof=1) / np.sqrt(n)             # standard error s / sqrt(n)
me = t * se                                 # margin of error

print(f"n = {n}, x_bar = {x.mean():.3f}, s = {x.std(ddof=1):.3f}")
print(f"t = {t:.3f}, standard error = {se:.4f}, margin of error = {me:.3f}")
print(f"{confidence:.0%} interval for mu: [{x.mean() - me:.2f}, {x.mean() + me:.2f}] hours")

**Your answer:**

### Task 2. The average basket value

RetailCorp's marketing team plans a campaign and wants to know the average basket value of all orders. Compute a 95%
confidence interval for the mean of `order_value_dkk`.

**Answer in words.** Report the result to the marketing team. Basket values are strongly skewed (Lecture 1). Can the
t interval still be used here? Explain why.

<details>
<summary>Hint 1</summary>

This is the interval for a mean with σ unknown, as in Task 1, on a different column. The t interval needs a normal
population or a large n.
</details>

<details>
<summary>Hint 2</summary>

`.mean()`, `.std(ddof=1)`, `len`, `np.sqrt` and `stats.t.ppf(0.975, df=n - 1)`. The first code pattern does all of it.
</details>

<details>
<summary>Hint 3</summary>

```python
# select the column order_value_dkk
# n, the mean and the standard deviation (ddof=1)
# the t critical value with n - 1 degrees of freedom
# the standard error and the margin of error
# print the lower and the upper limit
```
</details>

**Your answer:**

### Task 3. Orders that would qualify for free shipping

RetailCorp considers free shipping on orders above 1,000 DKK. What share of all orders would qualify? Compute a 95%
confidence interval for the share of orders with a basket value above 1,000 DKK, and check the condition.

**Answer in words.** In the period there were 50,000 orders. Translate the interval into a number of orders that
would have qualified, and write it for a manager.

<details>
<summary>Hint 1</summary>

A share is the mean of a 0/1 variable. Make one with the condition "basket value above 1,000", then use the interval
for a proportion. Check np̂(1 − p̂) > 5 first.
</details>

<details>
<summary>Hint 2</summary>

`(sample["order_value_dkk"] > 1000)` gives True and False; `.mean()` of it is the share. Then `stats.norm.ppf(0.975)`
and `np.sqrt`, as in the second code pattern.
</details>

<details>
<summary>Hint 3</summary>

```python
# make the condition: basket value above 1000
# n and p_hat (the mean of the condition)
# check the condition n * p_hat * (1 - p_hat)
# z, the standard error and the margin of error
# print the limits, then multiply them by 50,000
```
</details>

**Your answer:**

### Task 4. How predictable are deliveries from Aarhus?

The Aarhus warehouse wants to know how much single delivery times vary. Compute a 95% confidence interval for the
standard deviation σ of delivery times for orders shipped from Aarhus. The interval assumes a normal population, so
look at a histogram of the Aarhus delivery times first.

**Answer in words.** Is the assumption reasonable? What does the interval say about how predictable deliveries
from Aarhus are?

<details>
<summary>Hint 1</summary>

This is the interval for a variance, based on the chi-square distribution with n − 1 degrees of freedom. Select the
Aarhus orders first. Take the square root of both limits to get an interval for σ.
</details>

<details>
<summary>Hint 2</summary>

`sample.loc[sample["warehouse"] == "Aarhus", "delivery_hours"]`, `plt.hist`, `stats.skew`, `.var(ddof=1)`,
`stats.chi2.ppf(0.975, df=n - 1)` and `stats.chi2.ppf(0.025, df=n - 1)`, `np.sqrt`. See the third code pattern.
</details>

<details>
<summary>Hint 3</summary>

```python
# select the delivery times of the Aarhus orders
# histogram and skewness to check the normality assumption
# n and the sample variance (ddof=1)
# the two chi-square critical values
# lower = (n - 1) s^2 / larger value, upper = (n - 1) s^2 / smaller value
# print the limits for sigma^2 and their square roots for sigma
```
</details>

**Your answer:**

## Short problems

These problems use small numbers and no data. Solve them with the formulas from the slides, and use Python only to
look up critical values, or a table if you prefer. Write your reasoning in words, as you would explain it at the
exam.

### Problem 1. A t interval from summary numbers

A café chain measures the time it takes to serve 16 randomly chosen customers. The mean is 4.2 minutes and the
standard deviation 1.2 minutes. Serving times are roughly normal.

1. Compute a 95% confidence interval for the mean serving time.
2. Why do you use t and not z?
3. How would the interval change with 64 customers and the same mean and standard deviation?

<details>
<summary>Hint 1</summary>

σ is unknown, so use x̄ ± t<sub>n−1, α/2</sub> · s/√n with n − 1 = 15 degrees of freedom.
</details>

<details>
<summary>Hint 2</summary>

`stats.t.ppf(0.975, df=15)`, then the same with `df=63` for question 3.
</details>

**Your answer:**

### Problem 2. Planning a survey

A researcher wants to estimate the share of small and medium-sized firms in a region that use AI tools, within ±3
percentage points with 95% confidence. She has no idea what the share is.

1. How many firms must she survey?
2. How many for ±1.5 percentage points?
3. Explain why halving the margin of error does not just double the sample size.

<details>
<summary>Hint 1</summary>

n = 0.25 · z<sub>α/2</sub>² / ME², where 0.25 is the largest possible value of p̂(1 − p̂). Round up.
</details>

<details>
<summary>Hint 2</summary>

`stats.norm.ppf(0.975)` and `np.ceil`.
</details>

**Your answer:**

### Problem 3. A variance from a small sample

A machine fills bags of coffee. In a random sample of 10 bags the standard deviation of the weight is 3.0 g. The
weights are normally distributed.

1. Compute a 90% confidence interval for the standard deviation σ.
2. Would a 99% interval be wider or narrower? Check it.
3. Why does the problem need to say that the weights are normal?

<details>
<summary>Hint 1</summary>

(n − 1)s² = 9 × 3.0². Divide by the two chi-square values with 9 degrees of freedom, the larger one for the lower
limit. Take square roots for σ.
</details>

<details>
<summary>Hint 2</summary>

`stats.chi2.ppf(0.95, df=9)` and `stats.chi2.ppf(0.05, df=9)` for 90%; 0.995 and 0.005 for 99%.
</details>

**Your answer:**

## Extension tasks

### Extension 1. z or t for small samples?

Some people use z = 1.960 with s, also for small samples. Draw 2,000 random samples of n = 5 delivery times from the
population (seed 1). For each sample, compute a 95% interval with s and the t critical value, and one with s and
z = 1.960. Count how often each interval contains the population mean μ. Repeat with n = 30.

**Answer in words.** What do you find, and when does the choice between z and t matter?

<details>
<summary>Hint 1</summary>

Use the loop from the demonstration cell in the lecture notebook: draw a sample, compute both intervals, and count
when μ lies inside each of them.
</details>

<details>
<summary>Hint 2</summary>

`rng = np.random.default_rng(1)`, `rng.choice(pop, size=n, replace=False)`, `stats.t.ppf(0.975, df=n - 1)`, a
`for` loop and two counters.
</details>

**Your answer:**

### Extension 2. Sample size for the mean

How many orders does RetailCorp need to estimate the average delivery time within ±0.5 hours with 95% confidence?
Use σ = 7.82 hours (the formula assumes σ is known). Then draw one random sample of that size from the population
(seed 1) and compute the margin of error of its 95% t interval.

**Answer in words.** Is the margin of error close to 0.5? Why is it not exactly 0.5?

<details>
<summary>Hint 1</summary>

n = z<sub>α/2</sub>² · σ² / ME². Round up. Then compute a t interval for the new sample as in Task 1.
</details>

<details>
<summary>Hint 2</summary>

`stats.norm.ppf(0.975)`, `np.ceil`, `rng.choice(pop, size=n, replace=False)` and the first code pattern.
</details>

**Your answer:**